# 特徴抽出パイプライン

LLM特徴量 + 言語特徴量を抽出して `data/processed/features.csv` に保存する。

In [ ]:
import sys; sys.path.insert(0, '..')
import pandas as pd
from src.preprocessing.data_loader import load_dialogues
from src.preprocessing.text_normalizer import normalize
from src.preprocessing.tokenizer import pos_tag
from src.features.llm_features import extract_llm_features, OllamaClient
from src.features.linguistic_features import extract_linguistic_features
from src.features.feature_merger import merge_features

In [ ]:
# ローカルLLM = Ollama（別ターミナルで `ollama serve`、`ollama pull qwen2.5:7b` 済み前提）
# モデルは 'qwen2.5:7b' / 'llama3.1:8b' などに切替可
client = OllamaClient(model='qwen2.5:7b')

# 疎通確認（JSONで数値が返れば接続OK）
print(client.generate('Return ONLY a JSON object: {"ping": 1}'))

In [ ]:
dialogues = load_dialogues()  # デフォルト raw（英語）
llm_rows, ling_rows, labels = [], [], []
for i, d in enumerate(dialogues):
    # 文境界(. ? !)を保持した書き起こしをそのまま LLM・文長算出へ
    raw_text = d.text
    # POS/語彙特徴は正規化テキストをトークナイズ
    tagged = pos_tag(normalize(raw_text))
    llm_feat = extract_llm_features(raw_text, client.generate)
    ling_feat = extract_linguistic_features(raw_text, tagged)
    llm_rows.append(llm_feat.to_dict())
    ling_rows.append(ling_feat.to_dict())
    labels.append(d.label)
    print(f'\r{i + 1}/{len(dialogues)}', end='')

df = merge_features(llm_rows, ling_rows)
df['label'] = labels
df.to_csv('../data/processed/features.csv', index=False)
print('\n', df.shape)